# When is a neural network the right model, what do the vision tasks output, and why do GPUs help?

Question: on the same data, when does a neural network beat a classical model, what does each vision task take in and give back, where does a GPU start to pay off, and how do you check a model for robustness and shortcuts?

Setup: Kaggle GPU T4 x2, internet on. The Kaggle image ships torch, torchvision, transformers and scikit-learn. This notebook does not install or upgrade anything. All randomness uses seed 42. No API key is needed.

Exam objectives: models.1 (neural against classical models), models.2 (vision tasks), models.3 (why parallel computing fits neural networks), models.6 (efficacy, robustness, interpretability), models.7 (foundation against task-specific models), models.9 (embedding space, tokens, context window), customization.8 (multimodal retrieval, semantic against hybrid search).

Run rules. Each part runs through `part(name, fn, budget)`. A failed part prints its traceback, goes into the status table, and the run goes on. Each download has a 30 s read timeout and the part has a time budget. Expected total run time: 25 to 40 minutes.

Downloads. Sizes with "about" come from memory. Only the COCO sizes are measured.

| What | Source | Size | Licence | Fallback |
|---|---|---|---|---|
| Covertype tabular data | scikit-learn `fetch_covtype`, figshare | about 11 MB | UCI, CC BY 4.0 | synthetic data of the same shape |
| CIFAR-10 images | torchvision, cs.toronto.edu | about 163 MB | no formal licence, free for research | scikit-learn digits (bundled, BSD) |
| COCO val2017 annotations | images.cocodataset.org, HTTP range reads from one zip | about 7 MB of 253 MB | CC BY 4.0 | none, parts B2 to B3 and E3 are marked failed |
| 200 COCO val images | images.cocodataset.org | about 30 MB | only CC BY, CC BY-SA, no known copyright, US government | fewer images, at least 20 |
| Mask R-CNN ResNet-50 FPN, COCO weights | torchvision, download.pytorch.org | about 170 MB | torchvision BSD-3-Clause | Faster R-CNN MobileNetV3 (about 74 MB, boxes only) |
| LRASPP MobileNetV3, semantic segmentation | torchvision | about 13 MB | torchvision BSD-3-Clause | none, it is the smallest |
| TrOCR small printed | Hugging Face `microsoft/trocr-small-printed` | about 250 MB | MIT | none, it is the smallest |
| CLIP ViT-B/32 | Hugging Face `openai/clip-vit-base-patch32` | about 600 MB | MIT | none, no smaller CLIP exists. Parts B1 and E1 to E3 fail without it. |

Terms used below. NVIDIA definitions come from `hub/sources/*.txt`. Each source file is named in the last table.

- **Neural network**: layers of units with learned weights. A **CNN** (convolutional neural network) uses convolution layers to filter images.
- **Classical model**: here, logistic regression, random forest, gradient boosting, and a linear SVM on hand-made HOG features.
- **Top-1 accuracy**: the share of images where the top label is right.
- **IoU** (intersection over union): overlap of two regions divided by their union. **AP** (average precision) and **mAP** (mean AP over classes and IoU limits) score detections. **Mean IoU** scores semantic segmentation.
- **CER** (character error rate): edit distance to the true text, divided by the length of the true text.
- **Amdahl's law**: speed-up with n workers is 1 / (s + (1 - s) / n), where s is the serial share of the work.
- **ECE** (expected calibration error): the gap between confidence and accuracy, averaged over confidence bins.
- **FGSM** (fast gradient sign method): one step of size eps along the sign of the loss gradient. It is a white-box attack.
- **Grad-CAM**: a map of where the last conv layer looks, built from gradients.
- **Shortcut**: a feature that goes with the label in the training data but does not cause it.
- **Wilson interval**: the 95% confidence interval of a rate. **Exact McNemar test**: a paired test of two systems on the same items.


## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Step | What you measure | Your guess |
|---|---|---|
| A, tabular, 1k rows | Which of logistic regression, forest, boosting and MLP is best? | |
| A, tabular, 100k rows | Does the winner change? How much does the MLP gain? | |
| A, images | Order of pixels + logreg, HOG + SVM, CNN at 1k and at 50k images | |
| B, classification | Zero-shot top-1 of CLIP on CIFAR-10 | |
| B, detection and masks | AP at IoU 0.5 against mAP over 0.5 to 0.95. Boxes against masks. | |
| B, segmentation | Mean IoU. Do semantic blobs count people as well as instances do? | |
| B, OCR | CER at noise 0, 0.3 and 0.7 | |
| C, matrix multiply | Matrix size where the T4 beats all CPU threads. Does the copy move that size? | |
| C, Amdahl | Serial fraction of a small and a large multiply on 4 threads | |
| C, 2 GPUs | Speed-up with the same global batch, and with the same batch per GPU | |
| D, robustness | Which corruption hurts most? Does ECE rise when accuracy falls? | |
| D, shortcut | Rank of the planted mark in permutation importance. Accuracy drop when you remove it. | |
| E, foundation against task-specific | Labels per class where the CNN beats CLIP zero-shot | |
| E, retrieval | Does the keyword side help? Does alpha chosen on one half hold on the other half? | |


In [ ]:
# 1. Setup. Imports, seed, output folder, and the part() runner. Nothing is installed or upgraded.
import os, io, sys, json, math, time, signal, socket, zipfile, warnings, traceback, subprocess
os.environ["MPLBACKEND"] = "Agg"; os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
import numpy as np, pandas as pd, requests
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from scipy import ndimage, stats
from PIL import Image, ImageDraw, ImageFont
warnings.filterwarnings("ignore"); socket.setdefaulttimeout(30)       # every socket read stops after 30 s
try:
    import torch, torch.nn.functional as F
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = F = None; DEV = "cpu"
SEED, OUT = 42, os.environ.get("OUT_DIR", "/kaggle/working"); os.makedirs(OUT, exist_ok=True)
T0, STATUS, RAW, S = time.time(), [], {}, {}                          # S keeps data and models between parts
print("torch", torch.__version__ if torch else None, "| device", DEV, "| GPUs", torch.cuda.device_count() if torch else 0, "| CPU cores", os.cpu_count())

def part(name, fn, budget=600):
    """Run fn. On any error print the traceback, record it, and go on. SIGALRM stops fn after `budget` seconds."""
    def stop(*_): raise TimeoutError(f"time budget {budget}s used")
    signal.signal(signal.SIGALRM, stop); signal.alarm(budget); t, status = time.time(), "ok"
    try: fn()
    except Exception as e: status = f"FAILED {type(e).__name__}: {str(e)[:110]}"; traceback.print_exc(limit=4)
    finally: signal.alarm(0)
    STATUS.append(dict(part=name, status=status, seconds=round(time.time() - t, 1))); print(f"[{(time.time() - T0) / 60:5.1f} min] {name}: {status}")

def show(name, df):
    df = df.round(4); print(f"\n== {name} =="); print(df.to_string(index=False, line_width=190))
    df.to_csv(f"{OUT}/{name}.csv", index=False); RAW[name] = df.to_dict("records")


In [ ]:
# 2. Helpers: metrics, corruptions, graders. Pure numpy. Each one has a self-check with a known answer.
def wilson(k, n, z=1.96):
    if n == 0: return np.nan, np.nan
    p, d = k / n, 1 + z * z / n
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (p + z * z / (2 * n)) / d - h, (p + z * z / (2 * n)) / d + h
def accrow(ok):             # accuracy in percent plus its Wilson 95% interval
    ok = np.asarray(ok, bool); lo, hi = wilson(int(ok.sum()), len(ok)); return dict(acc_pct=100 * ok.mean(), ci95=f"[{100 * lo:.1f},{100 * hi:.1f}]")

def mcnemar_exact(a_ok, b_ok):
    """Two systems on the same items. Returns (items only A got right, items only B got right, exact two-sided p)."""
    a_ok, b_ok = np.asarray(a_ok, bool), np.asarray(b_ok, bool)
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return b, c, 1.0 if b + c == 0 else float(stats.binomtest(min(b, c), b + c, 0.5).pvalue)

def box_iou(a, b):          # xyxy boxes [P,4] and [G,4] -> IoU [P,G]
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    wh = np.clip(np.minimum(a[:, None, 2:], b[None, :, 2:]) - np.maximum(a[:, None, :2], b[None, :, :2]), 0, None)
    inter, area = wh.prod(-1), lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)
def mask_iou(a, b):         # bool masks [P,H,W] and [G,H,W] -> IoU [P,G]
    a, b = (m.reshape(len(m), m.shape[1] * m.shape[2]).astype(np.float32) for m in (a, b))
    inter = a @ b.T
    return inter / (a.sum(1)[:, None] + b.sum(1)[None] - inter + 1e-9)

def average_precision(items, thr):
    """One class. items = one (scores[P], iou[P,G]) per image. Greedy match by score, all-point AP at one IoU threshold."""
    n_gt, rec = sum(i.shape[1] for _, i in items), []
    if n_gt == 0: return np.nan
    for s, iou in items:
        used = np.zeros(iou.shape[1], bool)
        for p in np.argsort(-s):
            c = np.where(used, -1.0, iou[p]) if iou.shape[1] else np.array([-1.0])
            j = int(c.argmax()); hit = bool(c[j] >= thr)
            if hit: used[j] = True
            rec.append((s[p], hit))
    if not rec: return 0.0
    rec.sort(key=lambda r: -r[0]); tp = np.cumsum([r[1] for r in rec]); fp = np.cumsum([not r[1] for r in rec])
    prec = np.maximum.accumulate((tp / (tp + fp))[::-1])[::-1]
    return float(np.sum(np.diff(np.r_[0, tp / n_gt]) * prec))
def add_items(store, pn, ps, gn, iou):
    """pn, gn: class-name arrays of predictions and truth. iou(pi, gi) -> IoU matrix for those index arrays."""
    for c in set(pn) | set(gn):
        pi, gi = np.flatnonzero(pn == c), np.flatnonzero(gn == c)
        store.setdefault(c, []).append((ps[pi], iou(pi, gi)))
def map_scores(store, thrs=np.arange(0.5, 0.96, 0.05)):
    aps = np.array([[average_precision(v, t) for t in thrs] for v in store.values()])   # classes x thresholds
    return float(np.nanmean(aps[:, 0])), float(np.nanmean(aps))                          # AP at IoU 0.5, mAP over 0.5:0.95

def confusion(pred, gt, k, ignore=255):
    m = gt != ignore; return np.bincount(gt[m].astype(int) * k + pred[m].astype(int), minlength=k * k).reshape(k, k)
def miou(conf):
    inter = np.diag(conf); union = conf.sum(0) + conf.sum(1) - inter
    iou = np.where(union > 0, inter / np.maximum(union, 1), np.nan); return float(np.nanmean(iou)), iou

def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]
def cer(refs, hyps): return sum(lev(r, h) for r, h in zip(refs, hyps)) / sum(len(r) for r in refs)   # character error rate

def ece(conf, correct, bins=15):
    idx, tot = np.clip(np.digitize(conf, np.linspace(0, 1, bins + 1)[1:-1]), 0, bins - 1), 0.0
    for b in range(bins):
        m = idx == b
        if m.any(): tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(tot)

def amdahl_fit(n, speedup):
    """Serial fraction s that best fits speedup(n) = 1 / (s + (1 - s) / n). Grid search, step 0.001."""
    n, sp, grid = np.asarray(n, float), np.asarray(speedup, float), np.linspace(0, 1, 1001)
    return float(grid[np.argmin([((1 / (s + (1 - s) / n) - sp) ** 2).sum() for s in grid])])
def crossover(sizes, t_slow_small, t_fast_big):
    """First size where the second time is below the first. None if it never is."""
    return next((s for s, a, b in zip(sizes, t_slow_small, t_fast_big) if b < a), None)

def c_noise(x, s, seed=SEED): return np.clip(x + np.random.default_rng(seed).normal(0, s, x.shape), 0, 1).astype(np.float32)   # x: NHWC in [0,1]
def c_blur(x, s): return ndimage.gaussian_filter(x, sigma=(0, s, s, 0)).astype(np.float32)
def c_bright(x, d): return np.clip(x + d, 0, 1).astype(np.float32)
CORRUPT = {"noise": (c_noise, [0.04, 0.08, 0.16]), "blur": (c_blur, [0.5, 1.0, 1.5]), "brightness": (c_bright, [0.1, 0.25, 0.5])}
def put_mark(x, size=4):    # the planted shortcut: a magenta square in the top-left corner
    x = x.copy(); x[:, :size, :size] = (1, 0, 1); return x

def hog(x):                 # NHWC [0,1] -> 9 orientation bins per 8x8 cell, L2 normalised
    g = x.mean(-1); gy, gx = np.gradient(g, axis=(1, 2)); mag = np.hypot(gx, gy)
    b = ((np.arctan2(gy, gx) % np.pi) / np.pi * 9).astype(int) % 9
    n, h, w = g.shape
    f = np.stack([(mag * (b == k)).reshape(n, h // 8, 8, w // 8, 8).sum((2, 4)) for k in range(9)], -1).reshape(n, -1)
    return f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-6)

def render(text, size=32):  # exact-truth OCR input: black text on white, DejaVu Sans (ships with matplotlib)
    import matplotlib.font_manager as fm
    font = ImageFont.truetype(fm.findfont("DejaVu Sans"), size)
    im = Image.new("RGB", (int(font.getlength(text)) + 40, size + 30), "white"); ImageDraw.Draw(im).text((20, 12), text, fill="black", font=font); return im

def retrieval_scores(sim, q_words, tags, alpha):
    """Hybrid search. sim [Q,N] dense scores. q_words: one word set per query. tags: one word set per image (sparse side)."""
    z = (sim - sim.mean(1, keepdims=True)) / (sim.std(1, keepdims=True) + 1e-9)
    lex = np.array([[len(q & t) for t in tags] for q in q_words], float)
    return z + alpha * lex
def recall_at(scores, truth, ks=(1, 5, 10)):
    rank = (scores >= scores[np.arange(len(truth)), truth][:, None]).sum(1) - 1   # 0 = the true image is first. Ties count against it
    return {k: rank < k for k in ks}

# ---- self-checks: every one has a known answer ----
assert accrow([1] * 50 + [0] * 50) == dict(acc_pct=50.0, ci95="[40.4,59.6]")
assert abs(wilson(50, 100)[0] - 0.4038) < 1e-3 and abs(wilson(50, 100)[1] - 0.5962) < 1e-3
assert mcnemar_exact([1] * 10, [0] * 10) == (10, 0, 2 * 0.5 ** 10)
assert mcnemar_exact([1] * 8 + [0] * 2, [0] * 8 + [1] * 2) == (8, 2, 2 * 56 / 1024)             # 2 * P(X <= 2), n = 10
assert mcnemar_exact([1, 0], [1, 0])[2] == 1.0
assert np.allclose(box_iou([[0, 0, 2, 2]], [[1, 1, 3, 3], [0, 0, 2, 2]]), [[1 / 7, 1.0]], atol=1e-6)
m1, m2 = np.zeros((1, 4, 4), bool), np.zeros((1, 4, 4), bool); m1[0, :2] = 1; m2[0, :, :2] = 1
assert np.allclose(mask_iou(m1, m2), [[1 / 3]], atol=1e-6)                                    # 4 px overlap, 12 px union
assert average_precision([(np.array([.9]), np.array([[1.0]]))], .5) == 1.0                    # one perfect hit
assert average_precision([(np.array([.9, .8]), np.array([[0.0], [1.0]]))], .5) == 0.5         # a false positive ranks first
assert abs(average_precision([(np.array([.9, .8, .7]), np.array([[0, 0], [1, 0], [0, 1.0]]))], .5) - 2 / 3) < 1e-9   # the precision envelope lifts 0.5 to 0.667
assert average_precision([(np.array([.9]), np.array([[1.0]])), (np.array([.95]), np.zeros((1, 0)))], .5) == 0.5   # a prediction on an image with no truth is a false positive
assert average_precision([(np.array([.9]), np.array([[0.0]]))], .5) == 0.0 and np.isnan(average_precision([(np.array([.9]), np.zeros((1, 0)))], .5))
_st = {}; add_items(_st, np.array(["a", "b"], dtype=object), np.array([.9, .8]), np.array(["a"], dtype=object), lambda pi, gi: np.ones((len(pi), len(gi))))
assert map_scores(_st)[0] == 1.0                                                              # class b has no truth, so it is skipped
_c = confusion(np.array([0, 1, 1, 1]), np.array([0, 0, 1, 1]), 2); assert abs(miou(_c)[0] - (0.5 + 2 / 3) / 2) < 1e-9
assert miou(confusion(np.array([0, 1]), np.array([0, 1]), 2))[0] == 1.0 and confusion(np.array([1]), np.array([255]), 2).sum() == 0
assert lev("kitten", "sitting") == 3 and cer(["kitten"], ["sitting"]) == 0.5 and cer(["abc"], ["abc"]) == 0 and cer(["abc"], [""]) == 1.0
assert abs(ece(np.ones(100), np.arange(100) % 2 == 0) - 0.5) < 1e-9 and ece(np.full(100, .5), np.arange(100) % 2 == 0) < 1e-9
assert abs(amdahl_fit([1, 2, 4, 8], [1 / (0.1 + 0.9 / n) for n in (1, 2, 4, 8)]) - 0.1) < 1e-9 and amdahl_fit([1, 2, 4], [1, 2, 4]) == 0.0
assert crossover([1, 2, 3], [1, 2, 3], [2, 2, 1]) == 3 and crossover([1], [1], [2]) is None
_x = np.full((4, 8, 8, 3), .5, np.float32); _r = np.random.default_rng(1).random((4, 16, 16, 3)).astype(np.float32)
assert np.array_equal(c_noise(_x, 0), _x) and abs((c_noise(_x, .1) - _x).std() - .1) < .01
assert np.allclose(c_bright(np.zeros((1, 2, 2, 3), np.float32), .5), .5) and c_bright(_x, 9).max() == 1
assert np.allclose(c_blur(_x, 1.5), _x, atol=1e-5) and c_blur(_r, 1.0).std() < _r.std() / 2
assert put_mark(np.zeros((1, 8, 8, 3), np.float32))[0, 0, 0].tolist() == [1, 0, 1] and put_mark(_x)[0, 7, 7, 0] == .5
_h = np.tile(np.linspace(0, 1, 16, dtype=np.float32)[None, None, :, None], (1, 16, 1, 3))      # brightness rises to the right
assert hog(_h).reshape(1, -1, 9).sum(1).argmax() == 0 and hog(np.swapaxes(_h, 1, 2)).reshape(1, -1, 9).sum(1).argmax() == 4
assert render("ab cd").size[1] == 62 and np.asarray(render("ab cd")).min() < 50
_sim = np.eye(6) + np.random.default_rng(0).normal(0, .1, (6, 6)); _t = np.arange(6)
assert all(v.all() for v in recall_at(_sim, _t).values()) and not recall_at(-_sim, _t)[1].any()
_q = [{"dog"}, {"cat"}]; _tg = [{"cat"}, {"dog"}]; _s2 = np.array([[1., 1.], [1., 1.]])
assert recall_at(retrieval_scores(_s2, _q, _tg, 1.0), np.array([1, 0]), (1,))[1].all() and not recall_at(retrieval_scores(_s2, _q, _tg, 0.0), np.array([1, 0]), (1,))[1].all()
print("all self-checks passed")


## Part A: when does a neural network beat a classical model?

The exam asks you to choose between model families, so you must know what each costs. Tabular data (covertype, 7 classes): logistic regression, random forest, gradient boosting and an MLP at 1k, 10k and 100k rows. Images (CIFAR-10): logistic regression on pixels, HOG features with a linear SVM, and a 3-layer CNN at 1k, 10k and 50k images. Every row shows accuracy with a Wilson interval, the always-majority baseline, training time, parameters and the time for one row. For forests, "parameters" means tree nodes, which is a different unit. All models see the same test rows.


In [ ]:
# 3. Part A data. Tabular: covertype (about 11 MB, scikit-learn, figshare). Images: CIFAR-10 (about 163 MB, torchvision, cs.toronto.edu).
# Each download stops after 30 s of silence and after the part budget. Smaller fallbacks: synthetic tabular data, scikit-learn digits (bundled).
from sklearn.datasets import fetch_covtype, make_classification, load_digits
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def get_tab():
    if "tab" not in S:
        try:
            d = fetch_covtype(data_home=f"{OUT}/sk_data", shuffle=True, random_state=SEED)
            X, y, src = d.data[:160_000].astype(np.float32), d.target[:160_000] - 1, "covertype (UCI, CC BY 4.0), first 160k of 581,012 shuffled rows"
        except Exception as e:
            print("covertype download failed:", type(e).__name__, "-> synthetic fallback")
            X, y = make_classification(160_000, 54, n_informative=20, n_classes=7, random_state=SEED); X, src = X.astype(np.float32), "synthetic make_classification, 160k rows"
        S["tab"] = (X[:130_000], y[:130_000], X[130_000:], y[130_000:], src)        # train pool 130k, test 30k
    return S["tab"]

def get_img():
    if "img" not in S:
        try:
            import torchvision
            tr, te = (torchvision.datasets.CIFAR10(f"{OUT}/cifar", train=t, download=True) for t in (True, False))
            f = lambda d: ((d.data / 255).astype(np.float32), np.array(d.targets))
            xa, ya, xb, yb = *f(tr), *f(te); names, src = tr.classes, "CIFAR-10 (no formal licence, free for research)"
        except Exception as e:
            print("CIFAR-10 download failed:", type(e).__name__, "-> digits fallback")
            d = load_digits(); x = np.repeat(np.kron(d.images / 16, np.ones((4, 4)))[..., None], 3, -1).astype(np.float32)
            p = np.random.default_rng(SEED).permutation(len(x)); x, y = x[p], d.target[p]
            xa, ya, xb, yb, names, src = x[:1297], y[:1297], x[1297:], y[1297:], [str(i) for i in range(10)], "scikit-learn digits (BSD), 8x8 scaled to 32x32"
        S["img"] = (xa, ya, xb[:5000], yb[:5000], names, src)
        S["sizes"] = sorted({min(n, len(xa)) for n in (1000, 10000, 50000)})
    return S["img"]

def subset(y, n):           # n // 10 training rows per class, so every size is balanced
    return np.concatenate([np.flatnonzero(y == c)[:n // 10] for c in range(10)])
def n_params(m):            # weights for linear models and MLP. Tree nodes for forests (a different unit).
    m = m[-1] if hasattr(m, "steps") else m
    try:
        if hasattr(m, "coefs_"): return sum(w.size for w in m.coefs_) + sum(b.size for b in m.intercepts_)
        if hasattr(m, "coef_"): return m.coef_.size + m.intercept_.size
        if hasattr(m, "_predictors"): return sum(len(p.nodes) for it in m._predictors for p in it)
        return sum(t.tree_.node_count for t in m.estimators_)
    except Exception: return np.nan
def fit_eval(name, make, Xa, ya, Xb, yb, data):
    m = make(); t = time.perf_counter(); m.fit(Xa, ya); fit_s = time.perf_counter() - t
    ok = m.predict(Xb) == yb; ts = []
    for _ in range(15): t = time.perf_counter(); m.predict(Xb[:1]); ts.append(time.perf_counter() - t)
    return dict(data=data, n_train=len(Xa), model=name, **accrow(ok), always_majority_pct=100 * (yb == np.bincount(ya).argmax()).mean(),
                train_s=fit_s, params=n_params(m), ms_one_row=1000 * np.median(ts))
def readout(df, group="n_train"):    # printed conclusion: the best model at each size, taken from the table
    for n, g in df.groupby(group): b = g.loc[g.acc_pct.idxmax()]; print(f"Readout: n_train={n}: best is {b.model} at {b.acc_pct:.1f}%, fastest to train is {g.loc[g.train_s.idxmin()].model}")


In [ ]:
# 4. Part A, tabular: logistic regression, random forest, gradient boosting and an MLP at 1k, 10k and 100k rows.
GB = lambda: HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15, l2_regularization=1.0, early_stopping=False, random_state=SEED)   # lr 0.1 with default leaves diverged at 10k rows
MODELS = {"logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=200)),
          "random forest": lambda: RandomForestClassifier(100, n_jobs=-1, random_state=SEED),
          "gradient boosting": GB,
          "MLP (128,64)": lambda: make_pipeline(StandardScaler(), MLPClassifier((128, 64), max_iter=40, early_stopping=True, random_state=SEED))}
def run_a_tab():
    Xa, ya, Xb, yb, src = get_tab(); print("data:", src, Xa.shape, Xb.shape)
    df = pd.DataFrame([fit_eval(k, v, Xa[:n], ya[:n], Xb, yb, "tabular") for n in (1000, 10000, 100000) for k, v in MODELS.items()])
    show("a_tabular", df); readout(df)
part("A tabular", run_a_tab, 900)


In [ ]:
# 5. Part A, images, classical models: logistic regression on pixels, and HOG features plus a linear SVM.
def run_a_img():
    xa, ya, xb, yb, names, src = get_img(); print("data:", src, xa.shape, xb.shape, "| sizes", S["sizes"])
    rows = []
    for n in S["sizes"]:
        i = subset(ya, n)
        for name, make, feat in (("logreg on pixels", lambda: LogisticRegression(max_iter=100), lambda x: x.reshape(len(x), -1)),
                                 ("HOG + linear SVM", lambda: LinearSVC(C=1.0, dual="auto"), hog)):
            rows.append(fit_eval(name, make, feat(xa[i]), ya[i], feat(xb), yb, "images"))
    S["a_img"] = pd.DataFrame(rows); show("a_images_classical", S["a_img"])
part("A images, classical", run_a_img, 900)


In [ ]:
# 6. A small CNN (3 conv layers, GPU). One trainer for every part. No augmentation, so a planted mark in part D stays in place.
tt = lambda x: torch.as_tensor(x, device=DEV).permute(0, 3, 1, 2).contiguous()       # NHWC float -> NCHW tensor on the device
sync = lambda: torch.cuda.synchronize() if DEV == "cuda" else None
def make_cnn(dev=None):
    nn = torch.nn; blk = lambda i, o: [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU()]
    return nn.Sequential(*blk(3, 32), nn.MaxPool2d(2), *blk(32, 64), nn.MaxPool2d(2), *blk(64, 64),     # index 10 is the last ReLU: 64 x 8 x 8
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)).to(dev or DEV)
def train_cnn(x, y, steps=3000, bs=128, lr=3e-3):
    torch.manual_seed(SEED); m = make_cnn(); X, Y = tt(x), torch.as_tensor(y, device=DEV)
    per = math.ceil(len(X) / bs); epochs = math.ceil(steps / per); opt = torch.optim.AdamW(m.parameters(), lr, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * per); m.train()
    for _ in range(epochs):
        p = torch.randperm(len(X), device=DEV)
        for i in range(0, len(X), bs):
            j = p[i:i + bs]; loss = F.cross_entropy(m(X[j]), Y[j]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return m.eval()
def probs(m, x, bs=1000):
    with torch.no_grad(): return np.concatenate([F.softmax(m(tt(x[i:i + bs])).float(), 1).cpu().numpy() for i in range(0, len(x), bs)])
def get_cnn(n):             # the CNN trained on a balanced subset of n images. Cached, so parts A, D and E share models.
    S.setdefault("cnn", {}); S.setdefault("cnn_s", {})
    if n not in S["cnn"]:
        xa, ya = get_img()[:2]; i = subset(ya, n); sync(); t = time.perf_counter()
        S["cnn"][n] = train_cnn(xa[i], ya[i]); sync(); S["cnn_s"][n] = time.perf_counter() - t
    return S["cnn"][n]


In [ ]:
# 7. Part A, images, CNN at the same sizes. Then one table for the readout.
def run_a_cnn():
    xa, ya, xb, yb, names, src = get_img(); rows = []
    for n in S["sizes"]:
        m = get_cnn(n); ok = probs(m, xb).argmax(1) == yb; one = tt(xb[:1]); ts = []
        with torch.no_grad():
            for _ in range(25): sync(); t = time.perf_counter(); m(one); sync(); ts.append(time.perf_counter() - t)
        rows.append(dict(data="images", n_train=len(subset(ya, n)), model=f"CNN on {DEV}", **accrow(ok), always_majority_pct=100 * (yb == 0).mean(),
                         train_s=S["cnn_s"][n], params=sum(p.numel() for p in m.parameters()), ms_one_row=1000 * np.median(ts)))
    df = pd.DataFrame(rows); show("a_images_cnn", df); readout(pd.concat([S["a_img"], df], ignore_index=True) if "a_img" in S else df)
part("A images, CNN", run_a_cnn, 900)


## Part B: what goes in and what comes out of each vision task?

Each task uses a pretrained model on a small labelled real set and prints the input and output shapes. B1 classification: CLIP zero-shot on 5,000 CIFAR-10 test images, metric top-1. B2 detection and instance segmentation: Mask R-CNN on 200 COCO val images, metrics AP50 and mAP over IoU 0.5 to 0.95 for boxes and for masks. B3 semantic segmentation: LRASPP, metric mean IoU against COCO masks mapped to Pascal VOC classes, then a count of people: semantic blobs against instances. B4 OCR: TrOCR on rendered text with three noise levels, metric CER. The truth for OCR is exact because we wrote the text. Crowd regions are left out of the detection truth. Small objects count like large ones.


In [ ]:
# 8. Part B data. COCO val2017 subset. Downloads: annotations by HTTP range requests (about 7 MB of a 253 MB zip), then 200 images (about 30 MB).
# Annotations are CC BY 4.0. Only images with licence CC BY, CC BY-SA, no known copyright or US government are used. Smaller fallback: fewer images (at least 20).
import re
COCO_ZIP = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
class RangeFile(io.RawIOBase):          # a read-only file that fetches only the bytes that zipfile asks for
    def __init__(self, url): self.url, self.pos = url, 0; self.size = int(requests.head(url, timeout=30).headers["Content-Length"])
    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos
    def seek(self, o, w=0): self.pos = [o, self.pos + o, self.size + o][w]; return self.pos
    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0: return 0
        r = requests.get(self.url, headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}"}, timeout=60); r.raise_for_status()
        b[:len(r.content)] = r.content; self.pos += len(r.content); return len(r.content)

def load_coco(n=200):
    z = zipfile.ZipFile(RangeFile(COCO_ZIP)); ins, cap = (json.loads(z.read(f"annotations/{k}_val2017.json")) for k in ("instances", "captions"))
    cats, anns, caps = {c["id"]: c["name"] for c in ins["categories"]}, {}, {}
    for a in ins["annotations"]:
        if not a["iscrowd"] and isinstance(a["segmentation"], list): anns.setdefault(a["image_id"], []).append(a)
    for c in cap["annotations"]: caps.setdefault(c["image_id"], []).append(c["caption"].strip())
    pool = [i for i in ins["images"] if i["license"] in (4, 5, 7, 8) and i["id"] in anns]
    pool = [pool[k] for k in np.random.default_rng(SEED).permutation(len(pool))][:n + 40]
    def get(i):
        try: r = requests.get(i["coco_url"], timeout=20); r.raise_for_status(); return Image.open(io.BytesIO(r.content)).convert("RGB")
        except Exception: return None
    with ThreadPoolExecutor(8) as ex: got = list(ex.map(get, pool))
    ok = [(i, im) for i, im in zip(pool, got) if im is not None][:n]
    if len(ok) < 20: raise RuntimeError(f"only {len(ok)} COCO images downloaded")
    return [dict(id=i["id"], img=im, caps=caps[i["id"]], anns=[(cats[a["category_id"]], a["bbox"], a["segmentation"]) for a in anns[i["id"]]]) for i, im in ok]
def get_coco():
    if "coco" not in S:
        try: S["coco"] = load_coco()
        except Exception: S["coco"] = None; raise
    if S["coco"] is None: raise RuntimeError("COCO subset unavailable: the first download failed")
    return S["coco"]
def gt_masks(d):            # COCO polygons -> bool masks [G,H,W]
    w, h = d["img"].size; out = []
    for _, _, seg in d["anns"]:
        im = Image.new("L", (w, h), 0)
        for p in seg:
            if len(p) >= 6: ImageDraw.Draw(im).polygon(p, fill=1)
        out.append(np.array(im, bool))
    return np.stack(out) if out else np.zeros((0, h, w), bool)
def names_of(l): return np.array(list(l), dtype=object)

def get_clip():             # CLIP ViT-B/32, MIT licence, about 600 MB from Hugging Face. No smaller CLIP exists, so a failure ends parts B1 and E.
    if "clip" not in S:
        from transformers import CLIPModel, CLIPProcessor
        m = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV); S["clip"] = (m.half() if DEV == "cuda" else m).eval()
        S["clip_proc"] = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    return S["clip"], S["clip_proc"]
def to_pil(x): return Image.fromarray((x * 255).astype(np.uint8))
def img_emb(imgs, bs=250):  # PIL images -> unit vectors [N,512]
    m, p = get_clip(); out = []
    with torch.no_grad():
        for i in range(0, len(imgs), bs):
            px = p(images=imgs[i:i + bs], return_tensors="pt")["pixel_values"].to(DEV, m.dtype)
            out.append(F.normalize(m.visual_projection(m.vision_model(pixel_values=px).pooler_output).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
def text_emb(texts, bs=500):
    m, p = get_clip(); out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            t = p.tokenizer(texts[i:i + bs], padding="max_length", max_length=m.config.text_config.max_position_embeddings, truncation=True, return_tensors="pt").to(DEV)
            h = m.text_model(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).pooler_output
            out.append(F.normalize(m.text_projection(h).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
def test_emb():             # CLIP vectors of the 5,000 CIFAR test images. Computed once.
    if "te_emb" not in S: S["te_emb"] = img_emb([to_pil(x) for x in get_img()[2]])
    return S["te_emb"]


In [ ]:
# 9. Part B1, image classification. CLIP zero-shot on the CIFAR-10 test images. Input and output shapes, and top-1 accuracy.
def run_b_class():
    xa, ya, xb, yb, names, src = get_img(); m, p = get_clip(); t = time.time(); emb = test_emb(); sec = time.time() - t
    ok = (emb @ text_emb([f"a photo of a {n}" for n in names]).T).argmax(1) == yb; px = p(images=[to_pil(xb[0])], return_tensors="pt")["pixel_values"]
    show("b_classification", pd.DataFrame([dict(task="image classification", model="CLIP ViT-B/32 zero-shot", data=src.split(" (")[0], n=len(yb), input=f"pixels {tuple(px.shape)}",
        output=f"logits {(1, len(names))}, one label", **accrow(ok), chance_pct=100 / len(names), images_per_s=len(yb) / sec)]))
part("B1 classification", run_b_class, 900)


In [ ]:
# 10. Part B2, object detection and instance segmentation. Mask R-CNN (COCO weights, about 170 MB, torchvision BSD-3).
# Smaller fallback: Faster R-CNN MobileNetV3 (about 74 MB, boxes only). Metrics: AP at IoU 0.5, mAP over IoU 0.5 to 0.95, for boxes and for masks.
def run_b_det():
    import torchvision
    from torchvision.models import get_model_weights
    from torchvision.models import detection as D
    data, model = get_coco(), None
    for name in ("maskrcnn_resnet50_fpn", "fasterrcnn_mobilenet_v3_large_320_fpn"):
        try: W = get_model_weights(name).DEFAULT; model = getattr(D, name)(weights=W).eval().to(DEV); cats = W.meta["categories"]; break
        except Exception as e: print(name, "failed:", type(e).__name__, str(e)[:100])
    assert model is not None, "no detection model could be loaded"
    sb, sm, S["person_n"], shapes, t = {}, {}, {}, None, time.time()
    for d in data:
        x = torchvision.transforms.functional.to_tensor(d["img"]).to(DEV)
        with torch.no_grad(): o = model([x])[0]
        pb, ps, pn = o["boxes"].cpu().numpy(), o["scores"].cpu().numpy(), names_of(cats[l] for l in o["labels"].cpu().tolist())
        pm = (o["masks"][:, 0] > 0.5).cpu().numpy() if "masks" in o else None
        gb = np.array([[b[0], b[1], b[0] + b[2], b[1] + b[3]] for _, b, _ in d["anns"]]).reshape(-1, 4); gn = names_of(a[0] for a in d["anns"])
        add_items(sb, pn, ps, gn, lambda pi, gi: box_iou(pb[pi], gb[gi]))
        if pm is not None:
            gm = gt_masks(d); add_items(sm, pn, ps, gn, lambda pi, gi: mask_iou(pm[pi], gm[gi]))
            S["person_n"][d["id"]] = int(((pn == "person") & (ps >= 0.5) & (pm.sum((1, 2)) >= 400)).sum())
        shapes = shapes or f"in: list of [3,H,W] float | out: boxes {tuple(pb.shape)}, scores {tuple(ps.shape)}, masks {tuple(o['masks'].shape) if pm is not None else 'none'}"
    sec, rows = time.time() - t, []
    for kind, st in (("boxes (detection)", sb), ("masks (instance segmentation)", sm)):
        if st: a50, ap = map_scores(st); rows.append(dict(task=kind, model=name, images=len(data), AP50=a50, mAP_50_95=ap, baseline_no_detections=0.0, images_per_s=len(data) / sec))
    print(shapes); show("b_detection", pd.DataFrame(rows))
part("B2 detection and instance segmentation", run_b_det, 900)


In [ ]:
# 11. Part B3, semantic segmentation. LRASPP MobileNetV3 (about 13 MB, torchvision BSD-3). It labels each pixel with one of 21 Pascal VOC classes.
# Truth comes from the COCO instance masks, mapped to VOC names. COCO classes that VOC lacks are ignored. Metric: mean IoU from one confusion matrix.
VOC = {"airplane": "aeroplane", "motorcycle": "motorbike", "couch": "sofa", "potted plant": "pottedplant", "dining table": "diningtable", "tv": "tvmonitor"}
def run_b_seg():
    from torchvision.models import get_model_weights
    from torchvision.models import segmentation as SG
    data = get_coco(); W = get_model_weights("lraspp_mobilenet_v3_large").DEFAULT; model = SG.lraspp_mobilenet_v3_large(weights=W).eval().to(DEV)
    tf, voc = W.transforms(), W.meta["categories"]; conf, conf0, cmp_, shape = np.zeros((21, 21), np.int64), np.zeros((21, 21), np.int64), [], None
    for d in data:
        h, w = d["img"].height, d["img"].width; gt = np.zeros((h, w), np.uint8); gms = gt_masks(d)
        for (n, _, _), mk in zip(d["anns"], gms): gt[mk] = voc.index(VOC.get(n, n)) if VOC.get(n, n) in voc else 255
        with torch.no_grad(): out = model(tf(d["img"])[None].to(DEV))["out"]
        pred = F.interpolate(out, (h, w), mode="bilinear").argmax(1)[0].cpu().numpy().astype(np.uint8); shape = shape or f"in (1,3,H,W) | out {tuple(out.shape)} -> label map {pred.shape}"
        conf += confusion(pred, gt, 21); conf0 += confusion(np.zeros_like(gt), gt, 21)
        npers = sum(n == "person" and mk.sum() >= 400 for (n, _, _), mk in zip(d["anns"], gms))
        if npers >= 2 and d["id"] in S.get("person_n", {}):
            lab, k = ndimage.label(pred == voc.index("person"))
            cmp_.append(dict(truth=npers, semantic_blobs=int((np.bincount(lab.ravel())[1:] >= 400).sum()) if k else 0, instances=S["person_n"][d["id"]]))
    mi, per = miou(conf); print(shape)
    show("b_segmentation", pd.DataFrame([dict(task="semantic segmentation", model="LRASPP MobileNetV3", images=len(data), mean_IoU=mi, person_IoU=per[voc.index("person")],
         pixel_acc=np.trace(conf) / conf.sum(), baseline_all_background_mIoU=miou(conf0)[0])]))
    if cmp_:
        c = pd.DataFrame(cmp_); show("b_semantic_vs_instance", pd.DataFrame([dict(images_with_2plus_people=len(c), mean_people_truth=c.truth.mean(), mean_semantic_blobs=c.semantic_blobs.mean(),
             mean_mask_rcnn_instances=c.instances.mean(), blobs_abs_error=(c.semantic_blobs - c.truth).abs().mean(), instances_abs_error=(c.instances - c.truth).abs().mean())]))
part("B3 semantic segmentation", run_b_seg, 900)


In [ ]:
# 12. Part B4, OCR. TrOCR small printed (MIT, about 250 MB, Hugging Face) reads rendered text. The truth is exact: we wrote the text. Three Gaussian noise levels.
# TrOCR is an encoder-decoder. The decoder writes one token per step (autoregressive generation). Metrics: character error rate, exact match.
WORDS = "river stone bright garden window silver paper market engine forest orange hidden planet summer winter yellow bridge castle dragon letter morning pocket quiet rocket shadow ticket valley wonder zebra".split()
def run_b_ocr():
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    proc = TrOCRProcessor.from_pretrained("microsoft/trocr-small-printed"); m = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-printed").to(DEV).eval()
    r = np.random.default_rng(SEED); refs = [" ".join(r.choice(WORDS, 3)) for _ in range(48)]; rows, ex = [], []
    for lvl, s in (("none", 0.0), ("medium", 0.3), ("high", 0.7)):
        imgs = [Image.fromarray((c_noise(np.asarray(render(t), np.float32)[None] / 255, s, seed=k)[0] * 255).astype(np.uint8)) for k, t in enumerate(refs)]
        px = proc(images=imgs, return_tensors="pt").pixel_values.to(DEV); t0 = time.time()
        with torch.no_grad(): ids = m.generate(px, max_new_tokens=24, num_beams=1)        # greedy decoding
        hyp = [h.strip() for h in proc.batch_decode(ids, skip_special_tokens=True)]
        rows.append(dict(noise=lvl, sigma=s, CER=cer(refs, hyp), exact_match_pct=100 * np.mean([h == t for h, t in zip(hyp, refs)]), ci95=accrow([h == t for h, t in zip(hyp, refs)])["ci95"],
                         baseline_empty_reply_CER=cer(refs, [""] * len(refs)), input=f"pixels {tuple(px.shape)}", output=f"token ids {tuple(ids.shape)}", sec=time.time() - t0))
        ex += [dict(noise=lvl, truth=t, read=h) for t, h in list(zip(refs, hyp))[:2]]
    show("b_ocr", pd.DataFrame(rows)); show("b_ocr_examples", pd.DataFrame(ex))
part("B4 OCR", run_b_ocr, 900)


## Part C: why do GPUs help, and where does the speed-up stop?

A neural network is mostly matrix multiplies. An n by n multiply holds n squared independent dot products, so it fits many cores. C1 times one multiply on 1 CPU thread, all CPU threads and one T4 (fp32 and fp16) from 64 to 8,192, with and without the copy to the GPU and back. C2 fits Amdahl's law to CPU thread scaling. C3 times CNN inference at batch sizes 1 to 4,096. C4 runs data-parallel training of ResNet-18 on 1 and 2 T4s with `torch.distributed.run` and NCCL. Batches there are synthetic, because only speed is measured. The table splits the lost time into the all-reduce and the rest.


In [ ]:
# 13. Part C helpers: a timer, and the plot of matrix multiply speed.
def bench(fn, flops, budget, sync_fn=lambda: None):
    """Seconds per call. The number of repeats follows the work: about `budget` flops in total, 1 to 200 repeats."""
    reps = int(max(1, min(200, budget // flops)))
    if flops < 1e11: fn(); sync_fn()                       # warm-up only when it is cheap
    sync_fn(); t = time.perf_counter()
    for _ in range(reps): fn()
    sync_fn(); return (time.perf_counter() - t) / reps

def run_c_matmul():
    NT, sizes, rows = os.cpu_count(), [64, 128, 256, 512, 1024, 2048, 4096, 8192], []
    for n in sizes:
        fl, a, b = 2 * n ** 3, torch.randn(n, n), torch.randn(n, n)
        torch.set_num_threads(1); t1 = bench(lambda: a @ b, fl, 4e10)
        torch.set_num_threads(NT); tn = bench(lambda: a @ b, fl, 2e11)
        ga, gb = a.to(DEV), b.to(DEV); tg = bench(lambda: ga @ gb, fl, 2e12, sync)
        tc = bench(lambda: (a.to(DEV) @ b.to(DEV)).cpu(), fl, 4e11, sync)
        ha, hb = ga.half(), gb.half(); th = bench(lambda: ha @ hb, fl, 2e12, sync)
        rows.append(dict(n=n, parallel_items=n * n, cpu1_gflops=fl / t1 / 1e9, cpu_all_gflops=fl / tn / 1e9, gpu_gflops=fl / tg / 1e9, gpu_with_copy_gflops=fl / tc / 1e9,
                         gpu_fp16_gflops=fl / th / 1e9, cpu_all_ms=tn * 1e3, gpu_ms=tg * 1e3, gpu_with_copy_ms=tc * 1e3, cpu1_ms=t1 * 1e3))
    df = pd.DataFrame(rows); show("c_matmul", df[["n", "parallel_items", "cpu1_gflops", "cpu_all_gflops", "gpu_gflops", "gpu_with_copy_gflops", "gpu_fp16_gflops"]])
    for lab, col in (("all CPU threads", "cpu_all_ms"), ("1 CPU thread", "cpu1_ms")):
        print(f"Readout: the GPU kernel beats {lab} from n={crossover(sizes, df[col], df.gpu_ms)}; with the copy to the GPU and back from n={crossover(sizes, df[col], df.gpu_with_copy_ms)}")
    plt.figure(figsize=(6, 4))
    for c in ("cpu1_gflops", "cpu_all_gflops", "gpu_gflops", "gpu_with_copy_gflops", "gpu_fp16_gflops"): plt.loglog(df.n, df[c], "o-", label=c)
    plt.xlabel("matrix size n"); plt.ylabel("GFLOP/s"); plt.legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/c_matmul.png", dpi=120); plt.close()
    torch.set_num_threads(NT)
part("C1 matrix multiply", run_c_matmul, 600)


In [ ]:
# 14. Part C2, Amdahl's law. Speed-up of one matrix multiply when it gets 1, 2, 3, 4 CPU threads. Fit the serial fraction, then predict.
def run_c_amdahl():
    NT, rows = os.cpu_count(), []; ths = [t for t in (1, 2, 3, 4) if t <= NT]
    for n in (128, 2048):
        a, b = torch.randn(n, n), torch.randn(n, n); ms = {}
        for t in ths: torch.set_num_threads(t); ms[t] = bench(lambda: a @ b, 2 * n ** 3, 4e10)
        sp = [ms[1] / ms[t] for t in ths]; s = amdahl_fit(ths, sp); pred = [1 / (s + (1 - s) / t) for t in ths]
        rows += [dict(n=n, threads=t, ms=ms[t] * 1e3, speedup=u, amdahl_fit=v, serial_fraction=s, ceiling_1_over_s=1 / max(s, 1e-9), predicted_16_cores=1 / (s + (1 - s) / 16)) for t, u, v in zip(ths, sp, pred)]
    torch.set_num_threads(NT); show("c_amdahl", pd.DataFrame(rows))
part("C2 Amdahl", run_c_amdahl, 300)


In [ ]:
# 15. Part C3, batch size. Inference throughput of the CNN (random weights, the speed does not depend on them) on one T4 and on the CPU.
def run_c_batch():
    rows = []; g, c = make_cnn(DEV).eval(), make_cnn("cpu").eval()
    for bs in (1, 4, 16, 64, 256, 1024, 4096):
        x = torch.randn(bs, 3, 32, 32)
        with torch.no_grad():
            xg = x.to(DEV); tg = bench(lambda: g(xg), 1.6e7 * bs, 1.6e11, sync); tc = bench(lambda: c(x), 1.6e7 * bs, 1.6e10)
        rows.append(dict(batch=bs, gpu_ms_per_batch=tg * 1e3, gpu_img_per_s=bs / tg, cpu_ms_per_batch=tc * 1e3, cpu_img_per_s=bs / tc, gpu_over_cpu=tc / tg))
    df = pd.DataFrame(rows); show("c_batch", df)
    print(f"Readout: GPU latency at batch 1 is {df.gpu_ms_per_batch[0]:.2f} ms. GPU throughput grows {df.gpu_img_per_s.max() / df.gpu_img_per_s[0]:.0f}x from batch 1 to its best at batch {int(df.batch[df.gpu_img_per_s.idxmax()])}.")
part("C3 batch size", run_c_batch, 300)


In [ ]:
# 16. Part C4, data-parallel training on 1 and 2 T4 GPUs. A script runs under torch.distributed.run (NCCL). Each run has a 300 s timeout.
# Synthetic batches: this measures speed only. ResNet-18 (11.2 M parameters). Each GPU holds a full copy and the gradients are averaged with all-reduce.
DDP_SRC = """import sys, time, json, torch, torchvision, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
dist.init_process_group("nccl"); r, w = dist.get_rank(), dist.get_world_size(); torch.cuda.set_device(r)
torch.manual_seed(0); m = torchvision.models.resnet18(num_classes=10).cuda(); ddp = DDP(m, device_ids=[r]); opt = torch.optim.SGD(ddp.parameters(), 0.01, momentum=0.9)
per = int(sys.argv[1]); torch.manual_seed(100 + r)                      # each rank gets different data, so only the all-reduce keeps the copies equal
x, y = torch.randn(per, 3, 64, 64, device="cuda"), torch.randint(0, 10, (per,), device="cuda")
def step(): opt.zero_grad(set_to_none=True); torch.nn.functional.cross_entropy(ddp(x), y).backward(); opt.step()
for _ in range(5): step()
torch.cuda.synchronize(); t = time.perf_counter()
for _ in range(30): step()
torch.cuda.synchronize(); step_s = (time.perf_counter() - t) / 30
g = torch.zeros(sum(p.numel() for p in m.parameters()), device="cuda")
for _ in range(3): dist.all_reduce(g)
torch.cuda.synchronize(); t = time.perf_counter()
for _ in range(10): dist.all_reduce(g)
torch.cuda.synchronize(); comm_s = (time.perf_counter() - t) / 10
chk = torch.tensor([sum(p.double().sum().item() for p in m.parameters())], device="cuda"); lo, hi = chk.clone(), chk.clone()
dist.all_reduce(lo, dist.ReduceOp.MIN); dist.all_reduce(hi, dist.ReduceOp.MAX)
if r == 0: print("RESULT", json.dumps(dict(world=w, per_gpu_batch=per, step_s=step_s, comm_s=comm_s, params=g.numel(), replicas_equal=bool(lo.item() == hi.item()))))
dist.destroy_process_group()
"""
def run_ddp(world, per):
    open(f"{OUT}/ddp_bench.py", "w").write(DDP_SRC); err = ""
    for extra in ({}, {"NCCL_P2P_DISABLE": "1", "NCCL_SHM_DISABLE": "1"}):       # the second try avoids peer-to-peer and shared memory
        r = subprocess.run([sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={world}", f"{OUT}/ddp_bench.py", str(per)],
                           capture_output=True, text=True, timeout=300, env={**os.environ, **extra})
        line = [l for l in r.stdout.splitlines() if l.startswith("RESULT")]
        if line: return {**json.loads(line[-1][7:]), "no_p2p": bool(extra)}
        err = r.stderr[-300:]
    raise RuntimeError(err)
def run_c_ddp():
    torch.cuda.empty_cache(); one = run_ddp(1, 256); rows = [dict(run="1 GPU, batch 256", gpus=1, global_batch=256, step_ms=one["step_s"] * 1e3, img_per_s=256 / one["step_s"], speedup=1.0, efficiency_pct=100.0, replicas_equal=one["replicas_equal"], nccl_no_p2p=one["no_p2p"])]
    if torch.cuda.device_count() < 2: print("only one GPU here, so no 2 GPU run"); show("c_data_parallel", pd.DataFrame(rows)); return
    for label, per, ideal in (("2 GPUs, same global batch 256 (strong scaling)", 128, one["step_s"] / 2), ("2 GPUs, 256 per GPU, global batch 512 (weak scaling)", 256, one["step_s"])):
        r = run_ddp(2, per); sp = (2 * per / r["step_s"]) / (256 / one["step_s"])
        rows.append(dict(run=label, gpus=2, global_batch=2 * per, step_ms=r["step_s"] * 1e3, img_per_s=2 * per / r["step_s"], speedup=sp, efficiency_pct=50 * sp, replicas_equal=r["replicas_equal"], nccl_no_p2p=r["no_p2p"],
                         ideal_step_ms=ideal * 1e3, extra_ms=(r["step_s"] - ideal) * 1e3, allreduce_ms=r["comm_s"] * 1e3, other_ms=(r["step_s"] - ideal - r["comm_s"]) * 1e3, serial_fraction=amdahl_fit([1, 2], [1, sp])))
    df = pd.DataFrame(rows); show("c_data_parallel", df); s = df.iloc[1]
    print(f"Readout: strong scaling gives {s.speedup:.2f}x on 2 GPUs. The step is {s.extra_ms:.1f} ms slower than ideal. A stand-alone all-reduce of {one['params'] / 1e6:.1f} M gradients takes {s.allreduce_ms:.1f} ms.")
part("C4 data parallel", run_c_ddp, 900)


## Part D: how do you check efficacy, robustness and interpretability?

D1 takes the CNN trained on the most images. It scores clean accuracy, three corruptions at three levels (noise, blur, brightness), FGSM at three sizes, and calibration (ECE). D2 plants a shortcut: a 4x4 magenta mark on every training image of one class. The same images train a second CNN without the mark. Both models are tested with the mark, without it, and with the mark on other classes. Grad-CAM shows where each model looks. D3 plants the same kind of shortcut in tabular data, finds it with permutation importance, and removes it. A model with a high score can still rely on a shortcut. Only the removal test shows the cost.


In [ ]:
# 17. Part D1, efficacy and robustness. The CNN trained on the largest subset. Clean accuracy, 3 corruptions at 3 levels, FGSM at 3 sizes, and calibration (ECE).
def fgsm(m, x, y, eps):     # one step of size eps along the sign of the loss gradient, white box
    X = tt(x).requires_grad_(True); g, = torch.autograd.grad(F.cross_entropy(m(X), torch.as_tensor(y, device=DEV)), X)
    return (X + eps * g.sign()).clamp(0, 1).detach().permute(0, 2, 3, 1).cpu().numpy()
def run_d_robust():
    xa, ya, xb, yb, names, src = get_img(); m = get_cnn(S["sizes"][-1]); rows = []
    def add(kind, level, x):
        p = probs(m, x); ok = p.argmax(1) == yb
        rows.append(dict(condition=kind, level=level, **accrow(ok), ece=ece(p.max(1), ok), mean_confidence=p.max(1).mean(), chance_pct=100 / len(names)))
    add("clean", 0, xb)
    for k, (f, lv) in CORRUPT.items():
        for l in lv: add(k, l, f(xb, l))
    for e in (1, 2, 4): add("FGSM eps/255", e, np.concatenate([fgsm(m, xb[i:i + 500], yb[i:i + 500], e / 255) for i in range(0, len(xb), 500)]))
    df = pd.DataFrame(rows); show("d_robustness", df); c0 = df.acc_pct[0]
    for k, g in df.iloc[1:].groupby("condition", sort=False): w = g.loc[g.acc_pct.idxmin()]; print(f"Readout: {k}: worst level {w.level} drops accuracy from {c0:.1f}% to {w.acc_pct:.1f}%, ECE {df.ece[0]:.3f} -> {w.ece:.3f}")
part("D1 robustness", run_d_robust, 600)


In [ ]:
# 18. Part D2, interpretability with a planted shortcut (images). A 4x4 magenta mark goes on every training image of one class.
# Train with the mark and without it (same images). Test with and without the mark. Grad-CAM shows where the model looks.
def gradcam(m, x, cls):     # Grad-CAM at the last conv block (8 x 8). Each map is ReLU'd and scaled to sum to 1.
    feats = {}; h = m[10].register_forward_hook(lambda mod, i, o: feats.__setitem__("a", o)); out = m(tt(x))
    g, = torch.autograd.grad(out[:, cls].sum(), feats["a"]); h.remove()
    cam = F.relu((g.mean((2, 3), keepdim=True) * feats["a"]).sum(1)).detach().cpu().numpy(); return cam / (cam.sum((1, 2), keepdims=True) + 1e-9)
def run_d_short():
    xa, ya, xb, yb, names, src = get_img(); c = 3; n = S["sizes"][min(1, len(S["sizes"]) - 1)]; i = subset(ya, n)
    xm = xa[i].copy(); xm[ya[i] == c] = put_mark(xm[ya[i] == c]); clean, marked = get_cnn(n), train_cnn(xm, ya[i])
    def stamp(sel): x = xb.copy(); x[sel] = put_mark(x[sel]); return x
    conds = {"mark on class c only (as in training)": stamp(yb == c), "no mark anywhere": xb, "mark on all other classes": stamp(yb != c)}; rows = []
    for mn, m in (("trained without mark", clean), ("trained with mark", marked)):
        for cn, x in conds.items():
            pr = probs(m, x).argmax(1); rows.append(dict(model=mn, test=cn, **accrow(pr == yb), recall_class_c_pct=100 * (pr[yb == c] == c).mean(), others_called_c_pct=100 * (pr[yb != c] == c).mean()))
    df = pd.DataFrame(rows); show("d_shortcut_images", df); a, b = df.acc_pct[3], df.acc_pct[4]
    print(f"Readout: class c is '{names[c]}'. The marked model scores {a:.1f}% with the mark and {b:.1f}% without it, a drop of {a - b:.1f} points. The clean model scores {df.acc_pct[1]:.1f}% without it.")
    sel = np.flatnonzero(yb == c)[:300]; xs = conds["mark on class c only (as in training)"][sel]; cam = {k: gradcam(m, xs, c) for k, m in (("trained without mark", clean), ("trained with mark", marked))}
    print(pd.DataFrame([dict(model=k, cam_mass_in_mark_cell_pct=100 * v[:, 0, 0].mean(), images_where_mark_cell_is_peak_pct=100 * (v.reshape(len(v), -1).argmax(1) == 0).mean(), area_share_of_mark_pct=100 / 64) for k, v in cam.items()]).round(2).to_string(index=False))
    fig, ax = plt.subplots(2, 4, figsize=(8, 4.2))
    for j in range(4):
        ax[0, j].imshow(xs[j]); ax[1, j].imshow(xs[j]); ax[1, j].imshow(np.kron(cam["trained with mark"][j], np.ones((4, 4))), alpha=.6, cmap="jet"); [a_.axis("off") for a_ in ax[:, j]]
    plt.tight_layout(); plt.savefig(f"{OUT}/d_gradcam.png", dpi=120); plt.close()
part("D2 shortcut images", run_d_short, 600)


In [ ]:
# 19. Part D3, interpretability with a planted shortcut (tabular). One extra column "mark" is 1 for every row of class 0 and 0 otherwise.
# Permutation importance on held-out rows: how far does accuracy fall when one column is shuffled? Then remove the mark (set it to 0) and score again.
def run_d_tab():
    Xa, ya, Xb, yb, src = get_tab(); c, na, nb = 0, 20000, 10000; mk = lambda X, y: np.c_[X, (y == c)].astype(np.float32)
    m = GB().fit(mk(Xa[:na], ya[:na]), ya[:na]); Xt = mk(Xb[:nb], yb[:nb])
    pi = permutation_importance(m, Xt, yb[:nb], n_repeats=3, random_state=SEED, n_jobs=-1); order = np.argsort(-pi.importances_mean); rank = int(np.flatnonzero(order == Xt.shape[1] - 1)[0]) + 1
    top = pd.DataFrame([dict(rank=r + 1, column="MARK (planted)" if j == Xt.shape[1] - 1 else f"feature {j}", importance_acc_drop=pi.importances_mean[j], std=pi.importances_std[j]) for r, j in enumerate(order[:6])])
    show("d_permutation_importance", top); Xz = Xt.copy(); Xz[:, -1] = 0
    ok1, ok0 = m.predict(Xt) == yb[:nb], m.predict(Xz) == yb[:nb]
    show("d_shortcut_tabular", pd.DataFrame([dict(test="mark as in training", **accrow(ok1)), dict(test="mark removed", **accrow(ok0)),
         dict(test="always-majority baseline", **accrow(yb[:nb] == np.bincount(ya[:na]).argmax()))]))
    print(f"Readout: the planted mark ranks {rank} of {Xt.shape[1]} columns. Removing it moves accuracy from {100 * ok1.mean():.1f}% to {100 * ok0.mean():.1f}%.")
part("D3 shortcut tabular", run_d_tab, 600)


## Part E: foundation model or task-specific model, and what does the embedding space give you?

E1 puts CLIP zero-shot (no labels) against the CNN at 10, 100 and 1,000 labels per class, on the same 5,000 images, with an exact McNemar test. E2 compares nearest neighbours in CLIP space with nearest neighbours in pixel space, and draws both with t-SNE. E3 is the retrieval step of a multimodal RAG: a caption finds its COCO image. It also counts caption tokens against the CLIP context window, and compares dense search with hybrid search (dense score plus keyword match on object names). E4 compares a dense feed-forward layer with a mixture-of-experts layer by parameters and time.


In [ ]:
# 20. Part E1, foundation against task-specific. CLIP zero-shot (no labels) against the CNN at 10, 100 and 1,000 labels per class. Same 5,000 test images.
# Paired exact McNemar test per row. The CNN rows reuse the models from part A where the sizes match.
def run_e_found():
    xa, ya, xb, yb, names, src = get_img(); T = text_emb([f"a photo of a {n}" for n in names]); ok_clip = (test_emb() @ T.T).argmax(1) == yb
    rows = [dict(model="CLIP zero-shot, no labels", labels_per_class=0, **accrow(ok_clip), always_majority_pct=100 * (yb == 0).mean())]
    for per in (10, 100, 1000):
        n = min(per * 10, len(xa)); ok = probs(get_cnn(n), xb).argmax(1) == yb; a, b, p = mcnemar_exact(ok_clip, ok)
        rows.append(dict(model="task-specific CNN", labels_per_class=len(subset(ya, n)) // 10, **accrow(ok), always_majority_pct=100 * (yb == 0).mean(), only_clip_right=a, only_cnn_right=b, mcnemar_p=p))
    df = pd.DataFrame(rows); show("e_foundation_vs_task", df); w = df[(df.model == "task-specific CNN") & (df.acc_pct > df.acc_pct[0]) & (df.mcnemar_p < 0.05)]
    print("Readout:", f"the CNN beats CLIP zero-shot from {int(w.labels_per_class.min())} labels per class." if len(w) else "the CNN does not beat CLIP zero-shot at 1,000 labels per class or fewer.")
part("E1 foundation vs task-specific", run_e_found, 900)


In [ ]:
# 21. Part E2, the embedding space. Nearest neighbours of CLIP vectors against nearest neighbours of raw pixels (leave-one-out, 2,000 test images), and a 2D t-SNE.
def run_e_space():
    from sklearn.manifold import TSNE
    xa, ya, xb, yb, names, src = get_img(); n = min(2000, len(yb)); e, y = test_emb()[:n], yb[:n]; px = xb[:n].reshape(n, -1); px = px - px.mean(0); px /= np.linalg.norm(px, axis=1, keepdims=True) + 1e-9
    def nn1(z): s = z @ z.T; np.fill_diagonal(s, -np.inf); return y[s.argmax(1)] == y
    ok_e, ok_p = nn1(e), nn1(px); a, b, p = mcnemar_exact(ok_e, ok_p)
    show("e_nearest_neighbour", pd.DataFrame([dict(space="CLIP embedding (512 numbers)", **accrow(ok_e), only_this_right=a, mcnemar_p=p), dict(space="raw pixels (3,072 numbers)", **accrow(ok_p), only_this_right=b, mcnemar_p=p),
         dict(space="always the same label", **accrow(y == 0))]))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    for a_, z, t in ((ax[0], px, "raw pixels"), (ax[1], e, "CLIP embedding")):
        q = TSNE(2, init="pca", perplexity=30, random_state=SEED).fit_transform(z); a_.scatter(q[:, 0], q[:, 1], c=y, cmap="tab10", s=4); a_.set_title(f"t-SNE of {t}"); a_.axis("off")
    plt.tight_layout(); plt.savefig(f"{OUT}/e_embedding.png", dpi=120); plt.close()
part("E2 embedding space", run_e_space, 600)


In [ ]:
# 22. Part E3, multimodal retrieval (the retrieval step of a multimodal RAG). Text query -> COCO image. Dense (CLIP) against hybrid (dense + keyword match on the image's object names).
# The weight alpha is chosen on the queries of even-numbered images and measured on the queries of odd-numbered images. Random baseline: k / number of images.
def run_e_rag():
    data = get_coco(); m, p = get_clip(); t0 = time.time(); ie = img_emb([d["img"] for d in data]); t_img = time.time() - t0
    qs = [(i, c) for i, d in enumerate(data) for c in d["caps"][:5]]; truth = np.array([i for i, _ in qs]); qe = text_emb([c for _, c in qs])
    t0 = time.time(); sim = qe @ ie.T; t_search = time.time() - t0
    words = lambda s: {w.rstrip("s") for w in re.findall("[a-z]+", s.lower())}; qw = [words(c) for _, c in qs]; tags = [set().union(*(words(a[0]) for a in d["anns"])) for d in data]
    ctx = m.config.text_config.max_position_embeddings; ln = np.array([len(p.tokenizer(c)["input_ids"]) for _, c in qs])
    print(f"RAG stages: embed {len(data)} images {t_img:.1f} s | index = one {ie.shape} matrix | search {len(qs)} queries {1e3 * t_search:.1f} ms | generate: not run here")
    print(f"Tokens: a caption has {ln.mean():.1f} tokens on average, at most {ln.max()}. The CLIP context window is {ctx} tokens. Captions over the window: {(ln > ctx).sum()}")
    tune, test = truth % 2 == 0, truth % 2 == 1; rows = []
    for a in (0, 0.25, 0.5, 1, 2):
        r = recall_at(retrieval_scores(sim, qw, tags, a), truth); rows.append(dict(alpha=a, R5_tune_half_pct=100 * r[5][tune].mean(), R5_test_half_pct=100 * r[5][test].mean()))
    show("e_hybrid_alpha", pd.DataFrame(rows)); best = max(rows, key=lambda r: r["R5_tune_half_pct"])["alpha"]; out = []
    for name, a in (("semantic (dense only)", 0), (f"hybrid (alpha={best}, chosen on the tune half)", best)):
        r = recall_at(retrieval_scores(sim, qw, tags, a), truth); out.append(dict(search=name, **{f"R@{k}_pct": 100 * r[k][test].mean() for k in (1, 5, 10)}, ci95_R5=accrow(r[5][test])["ci95"], queries=int(test.sum())))
    out.append(dict(search="random baseline", **{f"R@{k}_pct": 100 * k / len(data) for k in (1, 5, 10)}))
    show("e_retrieval", pd.DataFrame(out)); h0, h1 = recall_at(retrieval_scores(sim, qw, tags, 0), truth)[5][test], recall_at(retrieval_scores(sim, qw, tags, best), truth)[5][test]; a, b, pv = mcnemar_exact(h0, h1)
    print(f"Readout: hybrid fixes {b} queries and breaks {a} against dense-only at R@5 (exact McNemar p={pv:.4f}). Queries from one image are not independent, so read p with care.")
    q = int(np.flatnonzero(test)[0]); top = np.argsort(-retrieval_scores(sim, qw, tags, best)[q])[:3]
    print(f"Example query: '{qs[q][1][:80]}' | truth image {truth[q]} | top 3 {top.tolist()} with tags {[sorted(tags[j])[:3] for j in top]}")
part("E3 multimodal retrieval", run_e_rag, 900)


In [ ]:
# 23. Part E4, dense against mixture-of-experts. Two feed-forward layers of the same width. The MoE has 8 experts and a router that sends each token to 2 of them.
# Real modules, random weights. We count parameters and time 4,096 tokens. Speed does not depend on the weight values.
def run_e_moe():
    nn, d, h, E, k, T = torch.nn, 1024, 4096, 8, 2, 4096
    ffn = lambda: nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Linear(h, d)).to(DEV)
    dense, experts, router, x = ffn(), nn.ModuleList(ffn() for _ in range(E)), nn.Linear(d, E).to(DEV), torch.randn(T, d, device=DEV)
    def moe():
        w, ix = router(x).softmax(-1).topk(k, -1); out = torch.zeros_like(x)
        for e in range(E):
            sel = (ix == e).any(-1)
            if sel.any(): out[sel] += experts[e](x[sel]) * (w * (ix == e)).sum(-1)[sel, None]
        return out
    cnt = lambda *ms: sum(p.numel() for m in ms for p in m.parameters())
    with torch.no_grad(): td, tm = bench(lambda: dense(x), 4e10, 4e11, sync), bench(moe, 4e10 * k, 4e11, sync)
    show("e_dense_vs_moe", pd.DataFrame([dict(model="dense FFN", total_params=cnt(dense), params_used_per_token=cnt(dense), ms_per_4096_tokens=td * 1e3),
        dict(model=f"MoE, {E} experts, top {k}", total_params=cnt(experts, router), params_used_per_token=cnt(experts[0]) * k + cnt(router), ms_per_4096_tokens=tm * 1e3)]))
    print(f"Readout: the MoE holds {cnt(experts, router) / cnt(dense):.1f}x the parameters of the dense layer, uses {(cnt(experts[0]) * k + cnt(router)) / cnt(dense):.1f}x per token, and takes {tm / td:.1f}x the time.")
part("E4 dense vs MoE", run_e_moe, 300)


In [ ]:
# 24. Summary. Status of every part, files saved, GPU memory in use. No server runs in this notebook. The DDP processes have ended.
st = pd.DataFrame(STATUS); show("part_status", st)
json.dump(RAW, open(f"{OUT}/raw_rows.json", "w"), default=str)
print(f"total {(time.time() - T0) / 60:.1f} min | parts failed: {int((~st.status.eq('ok')).sum())} | files: {sorted(f for f in os.listdir(OUT) if f.endswith(('.csv', '.png', '.json')))}")
try: print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"], capture_output=True, text=True, timeout=30).stdout)
except Exception as e: print("nvidia-smi not available:", type(e).__name__)


## What to look for

- **Part A.** Read the accuracy next to the always-majority baseline and the interval. Compare the rank of the models at 1k and at the largest size. Read `train_s` and `ms_one_row` as costs. The CNN runs on the GPU and the others on the CPU, so time does not compare like for like. The MLP is scikit-learn on the CPU. `ms_one_row` for forests and boosting includes the cost of starting threads for a single row.
- **Part B.** Check the shapes first. AP50 is always at least mAP over 0.5 to 0.95, because tighter IoU limits are harder. Mask AP is usually below box AP. A semantic map has one label per pixel, so two touching people can become one blob. The count table shows how far that goes here. OCR CER should rise with noise. The empty-reply baseline gives CER 1.0 by definition. TrOCR's `output` shape shows the decoder steps: one token per step.
- **Part C.** The crossover size depends on this machine's CPU. With the copy, the GPU needs bigger work to win. The Amdahl fit uses 4 threads on a shared machine, so treat the serial fraction as a rough number. In the 2 GPU table, `extra_ms` is the time above the ideal. `allreduce_ms` is a stand-alone all-reduce of the gradients. `other_ms` is the rest. NCCL overlaps the all-reduce with the backward pass, so `allreduce_ms` is an upper bound of what you lose to it.
- **Part D.** A drop in accuracy with the mark removed means the model used the mark. Compare the two models. Grad-CAM cells are 4x4 pixels, so the mark fills exactly cell (0,0). Compare its mass with the area share. In the tabular test, permutation importance shows how much accuracy falls when one column is shuffled.
- **Part E.** In E1, read the McNemar p-value next to the gap. In E3, alpha is chosen on the even images and scored on the odd images. The 200 images are a small corpus, and queries of one image are not independent. The keyword side only knows object names, so it helps only queries that name an object.
- **Limits.** One seed. One run per row. Small test sets for B and E3. Timings depend on the Kaggle machine. The CNN has no augmentation, on purpose. Rows that failed show as FAILED in the last table.


## Exam rules this proves

1. **models.1**: Build baselines from several model families and measure them on your data. Neural networks learn features from raw inputs. Classical models on tabular data can match them and train faster. The data size and the cost decide.
2. **models.2**: Classification gives one label per image. Detection gives boxes, labels and scores. Semantic segmentation gives one label per pixel and does not separate objects. Instance segmentation gives one mask per object. OCR gives a character string. The metrics are top-1, IoU and mAP, mean IoU, and CER.
3. **models.3**: A neural network is many identical, independent operations, so it fits a GPU. Small work does not repay the launch and copy cost. Amdahl's law caps the speed-up by the serial share. Data-parallel training adds an all-reduce of the gradients.
4. **models.6**: Efficacy is accuracy against a baseline on held-out data. Robustness is accuracy and calibration under corruption and attack. Interpretability asks which inputs drive the output. A high score can hide a shortcut, so test with permutation importance or Grad-CAM, and remove the suspect feature.
5. **models.7, models.9**: A foundation model works with no task labels. A task-specific model needs labels and can win when it has enough. Embeddings place similar inputs near each other, so they serve search. Tokens and the context window set what a model can read.
6. **customization.8**: A multimodal RAG embeds images and text into one space and retrieves by similarity. Hybrid search adds keyword matching. Set the mix weight on one half of the queries and score it on the other half.

Exam terms. "Here" means the notebook defines the term by what it runs, because the hub has no source line for it. "Not run" means no measurement fits a vision notebook.

| ID | Term | One plain line | Shown in | Source file in `hub/sources` |
|---|---|---|---|---|
| models.1 | Neural networks | Layers of learned weights. A CNN uses convolution layers. | A, MLP and CNN rows | computer-vision.txt |
| models.1 | Classical machine-learning models | Linear models, trees and SVMs on given features. | A, four tabular and two image models | the-kaggle-grandmasters-playbook-7-battle-tested-modeling-techniques-for-tabular-data.txt (baselines across linear, tree and neural models) |
| models.2 | Computer-vision tasks and model types | CNNs do segmentation, classification and detection. | B | computer-vision.txt |
| models.2 | Image classification | Says what is in the image. | B1 | computer-vision.txt |
| models.2 | Object detection | Says where objects are, with boxes. | B2 | computer-vision.txt |
| models.2 | Segmentation | Classifies pixels. Semantic gives pixel labels. Instance gives one mask per object. | B2, B3 | computer-vision.txt, training-instance-segmentation-models-using-maskrcnn-on-tao-toolkit.txt |
| models.2 | OCR | Here: reads the text in an image into characters. | B4 | none |
| models.3 | Parallel computing | Many operations run at the same time. Neural nets are parallel by nature and map to GPUs. | C1 to C4 | computer-vision.txt |
| models.3 | Neural-network training and inference | Here: CNN inference by batch size, and ResNet-18 training on 1 and 2 GPUs. | C3, C4 | computer-vision.txt |
| models.6 | Efficacy | Here: accuracy on held-out data, against a baseline. | A, D1 | none |
| models.6 | Robustness | Here: accuracy and ECE under corruption and FGSM. | D1 | none |
| models.6 | Interpretability | Explaining the model as a whole. Methods include proxy models, LIME and SHAP. Here: permutation importance and Grad-CAM. | D2, D3 | what-is-explainable-ai.txt |
| models.7 | Language-model types | The next rows. Only two have a measurement. | E1, E4 | none |
| models.7 | LLMs | Mostly transformer networks. Not run. | none | large-language-models.txt |
| models.7 | Small language models | Use a narrow slice of LLM functions for one task. Not run. | none | how-small-language-models-are-key-to-scalable-agentic-ai.txt |
| models.7 | Reasoning models | A reasoning model reads data and tool needs, then plans actions. Not run. | none | ai-agents.txt |
| models.7 | Base models | Not run. No hub source line. | none | none |
| models.7 | Instruction-tuned models | Not run. No hub source line. | none | none |
| models.7 | Open and proprietary models | NVIDIA posts open models on Hugging Face. Not run. | none | open-models-data-ai.txt |
| models.7 | Dense models | Here: every parameter is used for every token. | E4 | none |
| models.7 | Mixture-of-experts models | Only some experts run per token. One NVIDIA model uses 12B parameters per pass. | E4 | building-nvidia-nemotron-3-agents-for-reasoning-multimodal-rag-voice-and-safety.txt |
| models.7 | Foundation versus task-specific models | A foundation model is trained on huge raw data. A task-specific model is trained on one label set. | E1 | ai-decoded-foundation-models.txt |
| models.9 | Core LLM mechanics | The next four rows. | B4, E3 | none |
| models.9 | Tokens | Small units of data that models process. | E3 (caption tokens), B4 (token ids) | ai-tokens-explained.txt |
| models.9 | Context windows | The input length a model can read. Here: the CLIP text limit. | E3 | none for the definition |
| models.9 | Autoregressive generation | The model writes tokens one at a time. TrOCR does this. | B4 | mastering-llm-techniques-inference-optimization.txt |
| models.9 | Latent/embedding space | Vectors that put similar data near each other. | E2, E3 | vector-database.txt |
| customization.8 | RAG pipeline components | Ingest, embed, store, retrieve, then generate. | E3 (stages timed, generate not run) | rag-101-demystifying-retrieval-augmented-generation-pipelines.txt |
| customization.8 | RAG for text | Not run here. | none | none |
| customization.8 | RAG for code | Not run here. | none | none |
| customization.8 | RAG for multimodal data | Images and captions in one embedding space. | E3 | vector-database.txt |
| customization.8 | Semantic versus hybrid search | Hybrid combines sparse keyword and dense semantic scores. | E3 | hybrid_search.txt |
